# StateAwareXGBRanker — fixed-design LOPO robustness experiment

**Status:** implementation draft; no results have been run or inspected. This is an *additional* non-GNN baseline for the GraphEV study, not a replacement for the existing E6 experiment.

**Question:** Can explicit history-consistent, operation-local state features reproduce the ranking benefit of `DynamicReal` without graph message passing?

**Protocol:** Same 10 products, LOPO partitions, training-only categorical vocabularies, all future candidate operations per ranking state, post-current state convention, terminal singleton evaluation, E3 XGBRanker architecture/hyperparameters, seeds 17/29/43 and tie-aware evaluation. Compare the resulting product means against the **saved** E6 `dynamic_real` fold means, treating 10 held-out products as independent statistical units.

**Fixed state features (not tuned on held-out performance):** For the current and candidate operation, three endpoint slots (fixation source, fixation target, part node) each receive a 10-variable summary: active, removal-complete, effective remaining fixation degree, remaining degree / initial maximum, remaining / initial node degree, executed fixation count, executed / initial node degree, number of remaining fixation types, remaining fixation-type entropy, and leading-type share. For two-endpoint fixation operations, append sums/minima/maxima/absolute differences for remaining degree and remaining/initial ratio, plus the number of active and completed endpoints. Pair features concatenate current, candidate, and absolute differences of operation summaries. Four global prefix-derived summaries are included: active-node fraction, effective remaining-fixation fraction, removal-complete fraction, and executed-fixation fraction. The exact E3 static pairwise features are also retained.

**Important:** “Remaining fixation” means not yet executed *and with both endpoints structurally active*, exactly as the E6 state-update rule. Candidate identities merely locate their referenced nodes, as in the graph model; raw IDs are **not** input feature values. Future labels, future completion events, and held-out product labels are not used in feature generation. No post-hoc feature selection.

The source-code cells below embed the exact E3 and E6 definitions taken from GitHub commit `21b931fc3d7e05ec0ae01ad8b032016fb7706969`, so no online source download is required. Upload the external KIT dataset archive to `/content` as in the project's README, then run all cells. This notebook will **not** modify the public repository.

In [ ]:
# 1. Imports and frozen configuration

from pathlib import Path
from collections import Counter, defaultdict
import tarfile, json, math, random, time

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

SEEDS = [17, 29, 43]
SHUFFLE_IDS = [1, 2, 3, 4, 5]

EPOCHS = 30
NEGATIVES_PER_STATE = 8
LR = 2e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.10

EMBED_DIM = 12
HIDDEN_DIM = 64
GNN_DIM = 32

BASE_SEED = 20261003
GRAPH_SHUFFLE_BASE = 20261119

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_FITS = 10 * len(SEEDS)

print("Device:", DEVICE)
print("Torch:", torch.__version__)
print("Planned XGBRanker fits:", N_FITS)

if DEVICE.type != "cuda":
    print("WARNING: GPU is strongly recommended for this notebook.")

In [ ]:
# 2. Find the KIT dataset automatically.
#
# The cell first searches for already-extracted CSVs anywhere under /content.
# If not found, it searches for the uploaded KIT .tar archive and extracts it.

root = Path("/content")

catalogues = list(root.rglob("battery_catalogue.csv"))

if catalogues:
    DATA = catalogues[0].parent
else:
    archives = list(root.rglob("10.35097-emz24pksshndq468.tar"))
    if not archives:
        archives = list(root.rglob("10.35097-*.tar"))

    assert archives, (
        "KIT dataset not found. Upload 10.35097-emz24pksshndq468.tar "
        "to the Colab Files panel, then rerun this cell."
    )

    TAR_PATH = archives[0]
    print("Using archive:", TAR_PATH)

    work = root / "battery_disassembly_dataset_extracted"
    work.mkdir(parents=True, exist_ok=True)

    with tarfile.open(TAR_PATH, "r") as tf:
        tf.extractall(work)

    catalogues = list(work.rglob("battery_catalogue.csv"))
    assert catalogues, "battery_catalogue.csv not found after extraction."
    DATA = catalogues[0].parent

print("Dataset directory:", DATA)

In [ ]:
# 3. Load and validate raw data.

bat = pd.read_csv(DATA / "battery_catalogue.csv")
parts = pd.read_csv(DATA / "parts.csv")
fix = pd.read_csv(DATA / "fixations.csv")
ops = pd.read_csv(DATA / "operations.csv")

print("Batteries:", len(bat))
print("Parts:", len(parts))
print("Fixations:", len(fix))
print("Operations:", len(ops))

display(bat[["id", "name"]])

In [ ]:
# 4. Attach each operation to a battery and reconstruct the 10 reference chains.

battery_names = bat.set_index("id")["name"].to_dict()

fix_battery = fix.set_index("id")["battery_id"].to_dict()
part_battery = parts.set_index("id")["battery_id"].to_dict()
part_class_raw = parts.set_index("id")["class"].to_dict()

fix_info_raw = (
    fix.set_index("id")[["class", "fromPartID", "toPartID"]]
    .to_dict("index")
)

def operation_battery(row):
    if pd.notna(row.fixation_id):
        return int(fix_battery[int(row.fixation_id)])
    return int(part_battery[int(row.part_id)])

ops["battery_id"] = ops.apply(operation_battery, axis=1)
ops_by_id = ops.set_index("id")

chains = {}

for b in sorted(ops["battery_id"].unique()):
    sub = ops[ops["battery_id"] == b]
    ids = set(sub["id"].astype(int))
    next_ids = set(sub["next_op_id"].dropna().astype(int))

    starts = list(ids - next_ids)
    terminals = sub[sub["next_op_id"].isna()]["id"].astype(int).tolist()

    assert len(starts) == 1
    assert len(terminals) == 1

    seq = []
    seen = set()
    cur = int(starts[0])

    while cur in ids and cur not in seen:
        seen.add(cur)
        seq.append(cur)

        nxt = ops_by_id.loc[cur, "next_op_id"]
        if pd.isna(nxt):
            break
        cur = int(nxt)

    assert len(seq) == len(ids), f"Broken chain for battery {b}"
    chains[int(b)] = seq

print({
    battery_names[b]: len(seq)
    for b, seq in chains.items()
})

In [ ]:
# 5. Raw full relational signatures.
#
# These are used only for diagnostics (seen vs unseen motif/signature).
# They are NOT output classes for the neural ranking models.

def raw_full_signature(op_id):
    r = ops_by_id.loc[int(op_id)]
    skill = str(r.skill_id)

    if pd.notna(r.fixation_id):
        fi = fix_info_raw[int(r.fixation_id)]
        src_cls = str(part_class_raw[int(fi["fromPartID"])])
        dst_cls = str(part_class_raw[int(fi["toPartID"])])
        return (
            f"{skill}|fix:{fi['class']}|"
            f"{src_cls}->{dst_cls}"
        )

    return (
        f"{skill}|part:"
        f"{part_class_raw[int(r.part_id)]}"
    )

raw_signatures = {
    b: [raw_full_signature(op_id) for op_id in seq]
    for b, seq in chains.items()
}

In [ ]:
# 6. Fold-specific vocabularies.
#
# Vocabularies are learned ONLY from the 9 training batteries.
# Category 0 is <UNK>, so truly unseen test categories do not leak into training.

def make_vocab(values):
    vals = sorted({
        str(x)
        for x in values
        if pd.notna(x)
    })
    return {
        "<UNK>": 0,
        **{v: i + 1 for i, v in enumerate(vals)}
    }

def lookup(vocab, value):
    if pd.isna(value):
        return 0
    return vocab.get(str(value), 0)

def build_fold_vocab(train_batteries):
    return {
        "part": make_vocab(
            parts[
                parts["battery_id"].isin(train_batteries)
            ]["class"]
        ),
        "fix": make_vocab(
            fix[
                fix["battery_id"].isin(train_batteries)
            ]["class"]
        ),
        "skill": make_vocab(
            ops[
                ops["battery_id"].isin(train_batteries)
            ]["skill_id"]
        ),
    }

In [ ]:
# 7. Convert one battery into product-invariant graph + operation features.
#
# graph_mode:
#   "real"    -> true physical fixation topology
#   "shuffle" -> fixed permutation of topology relative to semantic/operation nodes
#   "noedge"  -> same GNN architecture, but no message-passing edges
#
# Operation-local attributes are identical across all graph modes.
# Therefore only the graph branch is manipulated.

def build_battery_pack(
    battery_id,
    vocab,
    graph_mode="real",
    shuffle_id=0,
):
    assert graph_mode in {"real", "shuffle", "noedge"}

    psub = parts[
        parts["battery_id"] == battery_id
    ].copy()

    fsub = fix[
        fix["battery_id"] == battery_id
    ].copy()

    part_id_to_local = {
        int(pid): i
        for i, pid in enumerate(psub["id"].astype(int))
    }

    n_nodes = len(psub)

    true_degree = np.zeros(n_nodes, dtype=np.float32)

    edge_src = []
    edge_dst = []
    edge_type = []

    for _, r in fsub.iterrows():
        a = part_id_to_local[int(r.fromPartID)]
        b = part_id_to_local[int(r.toPartID)]

        true_degree[a] += 1
        true_degree[b] += 1

        # Undirected message-passing representation.
        edge_src += [a, b]
        edge_dst += [b, a]

        et = lookup(vocab["fix"], r["class"])
        edge_type += [et, et]

    true_degree_norm = (
        true_degree / max(1.0, float(true_degree.max()))
    )

    node_part_class = np.array(
        [
            lookup(vocab["part"], cls)
            for cls in psub["class"]
        ],
        dtype=np.int64,
    )

    true_edge_index = np.array(
        [edge_src, edge_dst],
        dtype=np.int64,
    )

    true_edge_type = np.array(
        edge_type,
        dtype=np.int64,
    )

    if graph_mode == "real":
        graph_edge_index = true_edge_index.copy()
        graph_edge_type = true_edge_type.copy()
        graph_degree_norm = true_degree_norm.copy()

    elif graph_mode == "shuffle":
        # A deterministic, independent permutation for each
        # (shuffle_id, battery). This preserves the unlabeled graph,
        # but reassigns its physical locations to semantic nodes.
        rng_graph = np.random.default_rng(
            GRAPH_SHUFFLE_BASE
            + 10007 * int(shuffle_id)
            + int(battery_id)
        )

        permutation = rng_graph.permutation(n_nodes)

        graph_edge_index = permutation[true_edge_index]
        graph_edge_type = true_edge_type.copy()

        shuffled_degree = np.bincount(
            graph_edge_index.reshape(-1),
            minlength=n_nodes,
        ).astype(np.float32)

        graph_degree_norm = (
            shuffled_degree
            / max(1.0, float(shuffled_degree.max()))
        )

    else:  # noedge
        graph_edge_index = np.empty((2, 0), dtype=np.int64)
        graph_edge_type = np.empty((0,), dtype=np.int64)

        # Do not leak adjacency through the graph branch.
        graph_degree_norm = np.zeros(
            n_nodes,
            dtype=np.float32,
        )

    seq = [int(x) for x in chains[battery_id]]

    categorical = []
    graph_nodes = []
    numeric = []
    signatures = []

    for op_id in seq:
        r = ops_by_id.loc[op_id]

        skill_idx = lookup(
            vocab["skill"],
            r.skill_id,
        )

        if pd.notna(r.fixation_id):
            fi = fix_info_raw[int(r.fixation_id)]

            src_pid = int(fi["fromPartID"])
            dst_pid = int(fi["toPartID"])

            src_node = part_id_to_local[src_pid]
            dst_node = part_id_to_local[dst_pid]

            categorical.append([
                skill_idx,
                0,
                lookup(vocab["fix"], fi["class"]),
                lookup(vocab["part"], part_class_raw[src_pid]),
                lookup(vocab["part"], part_class_raw[dst_pid]),
                0,
            ])

            graph_nodes.append([
                src_node,
                dst_node,
                -1,
            ])

            # IMPORTANT:
            # identical operation-local features in every graph condition.
            numeric.append([
                true_degree_norm[src_node],
                true_degree_norm[dst_node],
                0.0,
            ])

        else:
            pid = int(r.part_id)
            node = part_id_to_local[pid]

            categorical.append([
                skill_idx,
                1,
                0,
                0,
                0,
                lookup(vocab["part"], part_class_raw[pid]),
            ])

            graph_nodes.append([
                -1,
                -1,
                node,
            ])

            numeric.append([
                0.0,
                0.0,
                true_degree_norm[node],
            ])

        signatures.append(raw_full_signature(op_id))

    return {
        "battery_id": int(battery_id),
        "battery_name": battery_names[int(battery_id)],
        "node_part_class": node_part_class,
        "node_degree": graph_degree_norm.astype(np.float32),
        "edge_index": graph_edge_index,
        "edge_type": graph_edge_type,
        "op_categorical": np.asarray(categorical, dtype=np.int64),
        "op_graph_nodes": np.asarray(graph_nodes, dtype=np.int64),
        "op_numeric": np.asarray(numeric, dtype=np.float32),
        "full_signatures": signatures,
        "n_ops": len(seq),
        "graph_mode": graph_mode,
        "shuffle_id": int(shuffle_id),
    }

In [ ]:
# 10. Tie-aware ranking metrics.

def expected_tie_metrics(scores, target_position=0, tol=1e-8):
    """
    Scores correspond to candidate operations.

    Returns the expected Hits@1, Hits@3, and reciprocal rank
    under random ordering within exact/near score ties.
    """

    scores = np.asarray(scores)
    target_score = scores[target_position]

    greater = int(
        np.sum(scores > target_score + tol)
    )

    equal = int(
        np.sum(
            np.abs(scores - target_score) <= tol
        )
    )

    equal = max(equal, 1)

    hits1 = (
        max(
            0,
            min(equal, 1 - greater),
        )
        / equal
        if greater < 1
        else 0.0
    )

    hits3 = (
        max(
            0,
            min(equal, 3 - greater),
        )
        / equal
        if greater < 3
        else 0.0
    )

    ranks = np.arange(
        greater + 1,
        greater + equal + 1,
        dtype=float,
    )

    mrr = float(
        np.mean(1.0 / ranks)
    )

    return hits1, hits3, mrr

In [ ]:
# 13. E3 dependencies and frozen boosted-ranker configuration

import sys
import subprocess
import itertools
import shutil
from collections import Counter, defaultdict, deque

try:
    import xgboost as xgb
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "xgboost"]
    )
    import xgboost as xgb

from sklearn.preprocessing import OneHotEncoder
from scipy import sparse
from packaging.version import Version

XGB_PARAMS = {
    "objective": "rank:pairwise",
    "eval_metric": "ndcg@3",
    "n_estimators": 250,
    "max_depth": 6,
    "learning_rate": 0.05,
    "min_child_weight": 1.0,
    "subsample": 0.90,
    "colsample_bytree": 0.90,
    "reg_lambda": 1.0,
    "reg_alpha": 0.0,
    "tree_method": "hist",
    "n_jobs": -1,
}

if torch.cuda.is_available():
    if Version(xgb.__version__) >= Version("2.0.0"):
        XGB_PARAMS["device"] = "cuda"
    else:
        XGB_PARAMS["tree_method"] = "gpu_hist"

CANDIDATE_BINS = [
    (1, 5, "1-5"),
    (6, 10, "6-10"),
    (11, 25, "11-25"),
    (26, 50, "26-50"),
    (51, 100, "51-100"),
    (101, 10**9, ">100"),
]

print("XGBoost:", xgb.__version__)
print("XGB parameters:", XGB_PARAMS)


In [ ]:
# 14. Fold-specific one-hot encoder and pair-feature matrix construction

def make_pair_encoder(vocab):
    n_skill = len(vocab["skill"])
    n_fix = len(vocab["fix"])
    n_part = len(vocab["part"])

    single_op_categories = [
        np.arange(n_skill),
        np.arange(2),
        np.arange(n_fix),
        np.arange(n_part),
        np.arange(n_part),
        np.arange(n_part),
    ]

    categories = single_op_categories + [
        x.copy() for x in single_op_categories
    ]

    kwargs = {
        "categories": categories,
        "handle_unknown": "ignore",
        "dtype": np.float32,
    }

    try:
        encoder = OneHotEncoder(
            sparse_output=True,
            **kwargs,
        )
    except TypeError:
        encoder = OneHotEncoder(
            sparse=True,
            **kwargs,
        )

    encoder.fit(
        np.zeros((1, 12), dtype=np.int64)
    )
    return encoder


def pair_feature_matrix(pack, t, candidate_indices, encoder):
    candidate_indices = np.asarray(
        candidate_indices,
        dtype=np.int64,
    )

    n = len(candidate_indices)

    cur_cat = np.repeat(
        pack["op_categorical"][t:t+1],
        n,
        axis=0,
    )
    cand_cat = pack["op_categorical"][candidate_indices]

    pair_cat = np.concatenate(
        [cur_cat, cand_cat],
        axis=1,
    )

    cat_sparse = encoder.transform(pair_cat)

    cur_num = np.repeat(
        pack["op_numeric"][t:t+1],
        n,
        axis=0,
    ).astype(np.float32)

    cand_num = (
        pack["op_numeric"][candidate_indices]
        .astype(np.float32)
    )

    abs_num_diff = np.abs(
        cur_num - cand_num
    ).astype(np.float32)

    same_cat = (
        cur_cat == cand_cat
    ).astype(np.float32)

    progress = np.full(
        (n, 1),
        t / max(1, pack["n_ops"] - 1),
        dtype=np.float32,
    )

    numeric = np.concatenate(
        [
            cur_num,
            cand_num,
            abs_num_diff,
            same_cat,
            progress,
        ],
        axis=1,
    ).astype(np.float32)

    return sparse.hstack(
        [
            cat_sparse,
            sparse.csr_matrix(numeric),
        ],
        format="csr",
        dtype=np.float32,
    )


def build_xgb_training_data(
    train_batteries,
    vocab,
):
    encoder = make_pair_encoder(vocab)

    X_parts = []
    y_parts = []
    qid_parts = []

    query_id = 0

    for b in train_batteries:
        pack = build_battery_pack(
            b,
            vocab,
            graph_mode="real",
            shuffle_id=0,
        )

        n_ops = pack["n_ops"]

        # Exclude the final singleton-only state.
        for t in range(n_ops - 2):
            candidate_indices = np.arange(
                t + 1,
                n_ops,
                dtype=np.int64,
            )

            Xq = pair_feature_matrix(
                pack,
                t,
                candidate_indices,
                encoder,
            )

            yq = np.zeros(
                len(candidate_indices),
                dtype=np.float32,
            )
            yq[0] = 1.0

            X_parts.append(Xq)
            y_parts.append(yq)
            qid_parts.append(
                np.full(
                    len(candidate_indices),
                    query_id,
                    dtype=np.int32,
                )
            )

            query_id += 1

    X = sparse.vstack(
        X_parts,
        format="csr",
        dtype=np.float32,
    )
    y = np.concatenate(y_parts)
    qid = np.concatenate(qid_parts)

    return X, y, qid, encoder


def build_xgb_model(seed):
    params = dict(XGB_PARAMS)
    params["random_state"] = int(seed)

    return xgb.XGBRanker(
        **params,
    )


In [ ]:
# 13. Dynamic state-graph construction

from collections import defaultdict

DYNAMIC_SHUFFLE_BASE = GRAPH_SHUFFLE_BASE


def _battery_structure_records(battery_id, vocab):
    psub = parts[
        parts["battery_id"] == battery_id
    ].copy()

    fsub = fix[
        fix["battery_id"] == battery_id
    ].copy()

    part_ids = psub["id"].astype(int).tolist()
    part_id_to_local = {
        pid: i
        for i, pid in enumerate(part_ids)
    }

    records = []

    for _, r in fsub.iterrows():
        records.append({
            "fixation_id": int(r["id"]),
            "u": part_id_to_local[int(r.fromPartID)],
            "v": part_id_to_local[int(r.toPartID)],
            "edge_type": lookup(vocab["fix"], r["class"]),
        })

    return (
        psub,
        part_id_to_local,
        records,
    )


def _dynamic_permutation(
    battery_id,
    n_nodes,
    shuffle_id,
):
    rng = np.random.default_rng(
        DYNAMIC_SHUFFLE_BASE
        + 10007 * int(shuffle_id)
        + int(battery_id)
    )

    return rng.permutation(
        n_nodes
    )


def _state_edges_and_degree(
    n_nodes,
    records,
    removed_fixations,
    active_nodes,
    initial_max_degree,
):
    edge_src = []
    edge_dst = []
    edge_type = []

    degree = np.zeros(
        n_nodes,
        dtype=np.float32,
    )

    for rec in records:
        if rec["fixation_id"] in removed_fixations:
            continue

        u = int(rec["u"])
        v = int(rec["v"])

        if not active_nodes[u] or not active_nodes[v]:
            continue

        degree[u] += 1.0
        degree[v] += 1.0

        edge_src += [u, v]
        edge_dst += [v, u]
        edge_type += [
            int(rec["edge_type"]),
            int(rec["edge_type"]),
        ]

    if len(edge_src):
        edge_index = np.asarray(
            [edge_src, edge_dst],
            dtype=np.int64,
        )
        edge_type = np.asarray(
            edge_type,
            dtype=np.int64,
        )
    else:
        edge_index = np.empty(
            (2, 0),
            dtype=np.int64,
        )
        edge_type = np.empty(
            (0,),
            dtype=np.int64,
        )

    degree_norm = (
        degree
        / max(
            1.0,
            float(initial_max_degree),
        )
    ).astype(np.float32)

    return (
        edge_index,
        edge_type,
        degree_norm,
    )


def _complete_state_edges_and_degree(
    active_nodes,
):
    active = np.flatnonzero(
        active_nodes
    ).astype(np.int64)

    n_nodes = len(
        active_nodes
    )

    if len(active) <= 1:
        return (
            np.empty(
                (2, 0),
                dtype=np.int64,
            ),
            np.empty(
                (0,),
                dtype=np.int64,
            ),
            np.zeros(
                n_nodes,
                dtype=np.float32,
            ),
        )

    src = np.repeat(
        active,
        len(active),
    )
    dst = np.tile(
        active,
        len(active),
    )

    keep = (
        src != dst
    )

    src = src[keep]
    dst = dst[keep]

    edge_index = np.asarray(
        [src, dst],
        dtype=np.int64,
    )

    # Edge-type 0 is the reserved <UNK>/dummy type.
    edge_type = np.zeros(
        len(src),
        dtype=np.int64,
    )

    degree_norm = np.zeros(
        n_nodes,
        dtype=np.float32,
    )

    # Complete-graph normalized degree among the currently active nodes.
    degree_norm[active] = (
        (len(active) - 1)
        / max(
            1,
            n_nodes - 1,
        )
    )

    return (
        edge_index,
        edge_type,
        degree_norm,
    )


def _has_remaining_incident_fixation(
    node,
    records,
    removed_fixations,
    active_nodes,
):
    """
    Causal check: does this currently active node still participate in any
    modeled fixation that has not yet been executed and whose other endpoint
    is also structurally active?
    """
    for rec in records:
        if rec["fixation_id"] in removed_fixations:
            continue

        u = int(rec["u"])
        v = int(rec["v"])

        if not active_nodes[u] or not active_nodes[v]:
            continue

        if node == u or node == v:
            return True

    return False


def _deactivate_completed_parts(
    records,
    removed_fixations,
    active_nodes,
    part_completed,
):
    """
    Deactivate a part only after:
      1) its part operation has already occurred, and
      2) it has no remaining active incident fixation.

    Iterate to a fixed point because deactivating one completed isolated node
    can make another completed node isolated as well.
    """
    changed = True

    while changed:
        changed = False

        pending = np.flatnonzero(
            part_completed
            & active_nodes
        )

        for node in pending:
            node = int(node)

            if not _has_remaining_incident_fixation(
                node=node,
                records=records,
                removed_fixations=removed_fixations,
                active_nodes=active_nodes,
            ):
                active_nodes[node] = False
                changed = True

    return active_nodes


def build_e6_state_pack(
    battery_id,
    vocab,
    condition,
    shuffle_id=0,
):
    """
    Conditions:
      static_real
      dynamic_real
      dynamic_noedge
      dynamic_complete
      dynamic_shuffle

    Returns the invariant operation pack plus one block-diagonal graph
    containing all post-current ranking states t=0,...,n_ops-2.
    """
    assert condition in {
        "static_real",
        "dynamic_real",
        "dynamic_noedge",
        "dynamic_complete",
        "dynamic_shuffle",
    }

    # Invariant operation features are always taken from the canonical
    # real-topology pack. Thus operation-local true degrees are identical
    # across every E6 graph condition.
    base_pack = build_battery_pack(
        battery_id,
        vocab,
        graph_mode="real",
        shuffle_id=0,
    )

    (
        psub,
        part_id_to_local,
        true_records,
    ) = _battery_structure_records(
        battery_id,
        vocab,
    )

    n_nodes = len(psub)
    n_states = base_pack["n_ops"] - 1

    initial_degree = np.zeros(
        n_nodes,
        dtype=np.float32,
    )

    for rec in true_records:
        initial_degree[rec["u"]] += 1.0
        initial_degree[rec["v"]] += 1.0

    initial_max_degree = max(
        1.0,
        float(initial_degree.max()),
    )

    if condition == "dynamic_shuffle":
        permutation = _dynamic_permutation(
            battery_id,
            n_nodes,
            shuffle_id,
        )

        records = [
            {
                "fixation_id": rec["fixation_id"],
                "u": int(permutation[rec["u"]]),
                "v": int(permutation[rec["v"]]),
                "edge_type": rec["edge_type"],
            }
            for rec in true_records
        ]

        # Part-completion events are transformed by the same permutation.
        event_node_map = permutation

    else:
        permutation = np.arange(
            n_nodes,
            dtype=np.int64,
        )

        records = [
            dict(rec)
            for rec in true_records
        ]

        event_node_map = permutation

    # Static-real template.
    static_edge_index, static_edge_type, static_degree = (
        _state_edges_and_degree(
            n_nodes=n_nodes,
            records=records,
            removed_fixations=set(),
            active_nodes=np.ones(
                n_nodes,
                dtype=bool,
            ),
            initial_max_degree=initial_max_degree,
        )
    )

    removed_fixations = set()
    active_nodes = np.ones(
        n_nodes,
        dtype=bool,
    )
    part_completed = np.zeros(
        n_nodes,
        dtype=bool,
    )

    state_edge_indices = []
    state_edge_types = []
    state_degrees = []
    state_active_fraction = []
    state_edge_fraction = []

    initial_undirected_edges = len(
        true_records
    )

    seq = [
        int(x)
        for x in chains[
            battery_id
        ]
    ]

    for t in range(
        n_states
    ):
        # State is POST-current-operation: first apply operation t.
        op_id = seq[t]
        r = ops_by_id.loc[
            op_id
        ]

        if condition != "static_real":
            if pd.notna(
                r.fixation_id
            ):
                removed_fixations.add(
                    int(
                        r.fixation_id
                    )
                )
            else:
                semantic_node = (
                    part_id_to_local[
                        int(r.part_id)
                    ]
                )

                structural_node = int(
                    event_node_map[
                        semantic_node
                    ]
                )

                part_completed[
                    structural_node
                ] = True

            # Safe causal deactivation: part nodes disappear only once their
            # own part operation has happened AND no modeled fixation remains.
            active_nodes = _deactivate_completed_parts(
                records=records,
                removed_fixations=removed_fixations,
                active_nodes=active_nodes,
                part_completed=part_completed,
            )

        if condition == "static_real":
            edge_index = (
                static_edge_index
            )
            edge_type = (
                static_edge_type
            )
            degree_norm = (
                static_degree
            )

            active_fraction = 1.0
            edge_fraction = 1.0

        elif condition == "dynamic_noedge":
            edge_index = np.empty(
                (2, 0),
                dtype=np.int64,
            )
            edge_type = np.empty(
                (0,),
                dtype=np.int64,
            )
            degree_norm = np.zeros(
                n_nodes,
                dtype=np.float32,
            )

            active_fraction = float(
                active_nodes.mean()
            )
            edge_fraction = 0.0

        elif condition == "dynamic_complete":
            (
                edge_index,
                edge_type,
                degree_norm,
            ) = _complete_state_edges_and_degree(
                active_nodes
            )

            active_fraction = float(
                active_nodes.mean()
            )

            max_directed = (
                n_nodes
                * max(
                    0,
                    n_nodes - 1,
                )
            )

            edge_fraction = (
                edge_index.shape[1]
                / max(
                    1,
                    max_directed,
                )
            )

        else:
            (
                edge_index,
                edge_type,
                degree_norm,
            ) = _state_edges_and_degree(
                n_nodes=n_nodes,
                records=records,
                removed_fixations=removed_fixations,
                active_nodes=active_nodes,
                initial_max_degree=initial_max_degree,
            )

            active_fraction = float(
                active_nodes.mean()
            )

            edge_fraction = (
                (edge_index.shape[1] / 2.0)
                / max(
                    1,
                    initial_undirected_edges,
                )
            )

        state_edge_indices.append(
            edge_index
        )
        state_edge_types.append(
            edge_type
        )
        state_degrees.append(
            degree_norm
        )
        state_active_fraction.append(
            active_fraction
        )
        state_edge_fraction.append(
            edge_fraction
        )

    # Build one block-diagonal graph containing every ranking state.
    block_src = []
    block_dst = []
    block_type = []

    for state_idx, (
        edge_index,
        edge_type,
    ) in enumerate(
        zip(
            state_edge_indices,
            state_edge_types,
        )
    ):
        offset = (
            state_idx
            * n_nodes
        )

        if edge_index.shape[1]:
            block_src.append(
                edge_index[0]
                + offset
            )
            block_dst.append(
                edge_index[1]
                + offset
            )
            block_type.append(
                edge_type
            )

    if block_src:
        block_edge_index = np.asarray(
            [
                np.concatenate(
                    block_src
                ),
                np.concatenate(
                    block_dst
                ),
            ],
            dtype=np.int64,
        )

        block_edge_type = np.concatenate(
            block_type
        ).astype(
            np.int64
        )
    else:
        block_edge_index = np.empty(
            (2, 0),
            dtype=np.int64,
        )

        block_edge_type = np.empty(
            (0,),
            dtype=np.int64,
        )

    block_pack = {
        "node_part_class": np.tile(
            base_pack[
                "node_part_class"
            ],
            n_states,
        ).astype(np.int64),
        "node_degree": np.concatenate(
            state_degrees
        ).astype(np.float32),
        "edge_index": block_edge_index,
        "edge_type": block_edge_type,
    }

    return {
        "base_pack": base_pack,
        "state_graph_pack": block_pack,
        "n_nodes": int(
            n_nodes
        ),
        "n_states": int(
            n_states
        ),
        "state_active_fraction": np.asarray(
            state_active_fraction,
            dtype=np.float32,
        ),
        "state_edge_fraction": np.asarray(
            state_edge_fraction,
            dtype=np.float32,
        ),
        "condition": condition,
        "shuffle_id": int(
            shuffle_id
        ),
        "permutation": permutation.astype(
            np.int64
        ),
    }


## Prefix-derived feature snapshots

The state at rank position `t` is built **after** applying operation `t`. The helpers below explicitly call the original E6 deactivation implementation. All state snapshots are constructed chronologically from the reference prefix; no future operation is applied to an earlier snapshot.

In [ ]:
# Frozen state-feature construction. The exact E6 deferred-deactivation
# helper is used rather than approximating it.
STATE_NODE_FEATURES = [
    "active", "removal_complete", "remaining_degree", "remaining_over_max_initial",
    "remaining_over_initial", "executed_degree", "executed_over_initial",
    "distinct_remaining_fixation_types", "remaining_type_entropy", "dominant_remaining_type_fraction",
]
GLOBAL_STATE_FEATURES = [
    "active_fraction", "effective_remaining_fixation_fraction",
    "removal_complete_fraction", "executed_fixation_fraction",
]


def state_snapshots_for_battery(battery_id, vocab):
    pack = build_battery_pack(battery_id, vocab, graph_mode="real", shuffle_id=0)
    _, part_id_to_local, records = _battery_structure_records(battery_id, vocab)
    n_nodes = len(part_id_to_local)
    n_ops = pack["n_ops"]
    n_types = len(vocab["fix"])
    initial_deg = np.zeros(n_nodes, dtype=np.float32)
    for rec in records:
        initial_deg[rec["u"]] += 1
        initial_deg[rec["v"]] += 1
    max_initial = max(1., float(initial_deg.max(initial=0)))
    denom_initial = np.maximum(initial_deg, 1.)
    total_edges = max(1, len(records))
    removed = set()
    active = np.ones(n_nodes, dtype=bool)
    completed = np.zeros(n_nodes, dtype=bool)
    node_states, global_states = [], []
    seq = [int(x) for x in chains[battery_id]]

    for t in range(n_ops - 1):
        r = ops_by_id.loc[seq[t]]
        if pd.notna(r.fixation_id):
            removed.add(int(r.fixation_id))
        else:
            node = part_id_to_local[int(r.part_id)]
            completed[node] = True
        active = _deactivate_completed_parts(
            records=records, removed_fixations=removed,
            active_nodes=active, part_completed=completed,
        )

        effective_deg = np.zeros(n_nodes, dtype=np.float32)
        executed_deg = np.zeros(n_nodes, dtype=np.float32)
        type_counts = np.zeros((n_nodes, n_types), dtype=np.float32)
        surviving_edges = 0
        for rec in records:
            u, v = int(rec["u"]), int(rec["v"])
            if rec["fixation_id"] in removed:
                executed_deg[u] += 1
                executed_deg[v] += 1
                continue
            if not active[u] or not active[v]:
                continue
            effective_deg[u] += 1
            effective_deg[v] += 1
            type_counts[u, int(rec["edge_type"])] += 1
            type_counts[v, int(rec["edge_type"])] += 1
            surviving_edges += 1

        counts_sum = type_counts.sum(axis=1)
        prob = np.divide(type_counts, counts_sum[:, None],
                         out=np.zeros_like(type_counts), where=counts_sum[:, None] > 0)
        entropy = -(np.where(prob > 0, prob * np.log(np.maximum(prob, 1e-12)), 0.)).sum(axis=1)
        entropy /= np.log(max(2, n_types))
        leading_fraction = prob.max(axis=1)
        diverse = (type_counts > 0).sum(axis=1).astype(np.float32)
        node_matrix = np.column_stack([
            active.astype(np.float32), completed.astype(np.float32),
            effective_deg, effective_deg / max_initial,
            effective_deg / denom_initial, executed_deg,
            executed_deg / denom_initial, diverse, entropy, leading_fraction,
        ]).astype(np.float32)
        global_vector = np.array([
            active.mean(), surviving_edges / total_edges,
            completed.mean(), len(removed) / total_edges,
        ], dtype=np.float32)
        assert np.isfinite(node_matrix).all() and np.isfinite(global_vector).all()
        assert np.isclose(effective_deg.sum(), 2 * surviving_edges)
        node_states.append(node_matrix)
        global_states.append(global_vector)

    return {"pack": pack, "nodes": node_states,
            "global": np.stack(global_states).astype(np.float32),
            "node_feature_names": STATE_NODE_FEATURES}


def operation_state_features(pack, node_features, operation_indices):
    idx = np.asarray(operation_indices, dtype=np.int64)
    slots = pack["op_graph_nodes"][idx]  # source, target, part
    n = len(idx)
    k = node_features.shape[1]
    feature_slots = []
    for slot_idx in range(3):
        buf = np.zeros((n, k), dtype=np.float32)
        valid = slots[:, slot_idx] >= 0
        if np.any(valid):
            buf[valid] = node_features[slots[valid, slot_idx]]
        feature_slots.append(buf)
    a, b = feature_slots[0], feature_slots[1]
    # Endpoint aggregates only apply to fixation operations (the two slots
    # are zero for part operations, so the values are then naturally zero).
    aggregates = []
    for j in [2, 4]:  # remaining count; remaining / initial count
        x, y = a[:, j:j+1], b[:, j:j+1]
        aggregates.extend([x+y, np.minimum(x,y), np.maximum(x,y), np.abs(x-y)])
    aggregates.extend([a[:, 0:1] + b[:, 0:1], a[:, 1:2] + b[:, 1:2]])
    return np.concatenate(feature_slots + aggregates, axis=1).astype(np.float32)


def stateaware_pair_matrix(state_pack, t, candidate_indices, encoder):
    base = state_pack["pack"]
    ids = np.asarray(candidate_indices, dtype=np.int64)
    static_features = pair_feature_matrix(base, t, ids, encoder)
    state = state_pack["nodes"][t]
    current = operation_state_features(base, state, [t])
    candidate = operation_state_features(base, state, ids)
    current_repeated = np.repeat(current, len(ids), axis=0)
    global_repeated = np.repeat(state_pack["global"][t:t+1], len(ids), axis=0)
    extra = np.concatenate([
        current_repeated, candidate, np.abs(current_repeated-candidate), global_repeated
    ], axis=1).astype(np.float32)
    assert extra.shape[1] == 3 * current.shape[1] + len(GLOBAL_STATE_FEATURES)
    assert np.isfinite(extra).all()
    return sparse.hstack([static_features, sparse.csr_matrix(extra)],
                         format="csr", dtype=np.float32)

## Implementation checks before any full fitting

The E6 state builder and the new feature builder must agree on the effective remaining-edge and active-node fractions. This test does not look at prediction results.

In [ ]:
# Do not skip this pre-fit check.
CHECK_BATTERY = sorted(chains)[0]
check_train = [b for b in sorted(chains) if b != CHECK_BATTERY]
check_vocab = build_fold_vocab(check_train)
new_states = state_snapshots_for_battery(CHECK_BATTERY, check_vocab)
e6_states = build_e6_state_pack(CHECK_BATTERY, check_vocab, "dynamic_real")
np.testing.assert_allclose(new_states["global"][:, 0],
                           e6_states["state_active_fraction"], atol=1e-7, rtol=0)
np.testing.assert_allclose(new_states["global"][:, 1],
                           e6_states["state_edge_fraction"], atol=1e-7, rtol=0)
check_encoder = make_pair_encoder(check_vocab)
check_n = new_states["pack"]["n_ops"]
for t in [0, check_n // 2, check_n - 2]:
    q = np.arange(t+1, check_n)
    X = stateaware_pair_matrix(new_states, t, q, check_encoder)
    assert X.shape[0] == len(q) and np.isfinite(X.data).all()
print("PASS: post-current states match canonical E6 fractions; feature matrices valid.")
print("Smoke-check feature dimension:", X.shape[1])

In [ ]:
# Extended integrity audit: run AFTER original cells 1–13 and BEFORE training.
# This cell does not fit models and does not change any features or parameters.
validated_products = []
for held_out in sorted(chains):
    training_b = [b for b in sorted(chains) if b != held_out]
    audit_vocab = build_fold_vocab(training_b)
    new = state_snapshots_for_battery(held_out, audit_vocab)
    ref = build_e6_state_pack(held_out, audit_vocab, condition="dynamic_real")
    nstates = new["pack"]["n_ops"] - 1
    nnodes = ref["n_nodes"]
    assert nstates == ref["n_states"] == len(new["nodes"])
    np.testing.assert_allclose(new["global"][:, 0],
                               ref["state_active_fraction"], rtol=0, atol=1e-7)
    np.testing.assert_allclose(new["global"][:, 1],
                               ref["state_edge_fraction"], rtol=0, atol=1e-7)
    # Most informative check: compare the full per-node remaining-degree
    # trajectories, not just average fractions.
    ref_degrees = ref["state_graph_pack"]["node_degree"].reshape(nstates, nnodes)
    feature_degrees = np.stack([state[:, 3] for state in new["nodes"]])
    np.testing.assert_allclose(feature_degrees, ref_degrees,
                               rtol=0, atol=1e-7)
    for t in sorted(set([0, nstates // 2, nstates - 1])):
        q = np.arange(t + 1, new["pack"]["n_ops"], dtype=np.int64)
        matrix = stateaware_pair_matrix(new, t, q, make_pair_encoder(audit_vocab))
        assert matrix.shape[0] == len(q)
        assert np.isfinite(matrix.data).all()
    validated_products.append((held_out, battery_names[held_out], nstates, nnodes))
print('PASS: all', len(validated_products),
      'products match E6 on active fractions, effective edge fractions,',
      'and EVERY node degree at EVERY ranking state.')

# Future-order perturbation: permute only operations AFTER the current decision.
# Candidate feature rows, matched by operation identity, must remain identical.
held_out = sorted(chains)[0]
training_b = [b for b in sorted(chains) if b != held_out]
audit_vocab = build_fold_vocab(training_b)
encoder = make_pair_encoder(audit_vocab)
original_seq = list(chains[held_out])
t = max(1, min(12, len(original_seq) - 3))
base = state_snapshots_for_battery(held_out, audit_vocab)
orig_ids = original_seq[t+1:]
X_base = stateaware_pair_matrix(base, t,
                np.arange(t+1, len(original_seq), dtype=np.int64), encoder)
try:
    chains[held_out] = original_seq[:t+1] + list(reversed(original_seq[t+1:]))
    perturb = state_snapshots_for_battery(held_out, audit_vocab)
    np.testing.assert_allclose(base['nodes'][t], perturb['nodes'][t], rtol=0, atol=1e-7)
    np.testing.assert_allclose(base['global'][t], perturb['global'][t], rtol=0, atol=1e-7)
    changed_ids = chains[held_out][t+1:]
    X_changed = stateaware_pair_matrix(perturb, t,
                     np.arange(t+1, len(original_seq), dtype=np.int64), encoder)
    row_of = {op_id:i for i,op_id in enumerate(changed_ids)}
    X_reindexed = X_changed[[row_of[op] for op in orig_ids], :]
    np.testing.assert_allclose(X_base.toarray(), X_reindexed.toarray(),
                               rtol=0, atol=1e-7)
    print('PASS: current-state and candidate features are unchanged when the FUTURE operation order is permuted.')
finally:
    chains[held_out] = original_seq
print('Integrity checks complete. Results are not independently rerun by this validation cell.')


## Fixed training and LOPO evaluation

The candidate set for each query is the complete reference suffix. The terminal singleton state is excluded from training (no negatives) and included in evaluation. The original static E3 XGBRanker parameters and tie-aware metric implementation are reused unchanged. There is no held-out-product-based tuning or feature selection.

In [ ]:
def build_stateaware_training_data(training_products, vocab):
    encoder = make_pair_encoder(vocab)
    X_parts, y_parts, qid_parts = [], [], []
    query_id = 0
    for battery_id in training_products:
        state_pack = state_snapshots_for_battery(battery_id, vocab)
        n_ops = state_pack["pack"]["n_ops"]
        for t in range(n_ops - 2):
            suffix = np.arange(t+1, n_ops, dtype=np.int64)
            X_parts.append(stateaware_pair_matrix(state_pack, t, suffix, encoder))
            y = np.zeros(len(suffix), dtype=np.float32)
            y[0] = 1.
            y_parts.append(y)
            qid_parts.append(np.full(len(suffix), query_id, dtype=np.int32))
            query_id += 1
    return (sparse.vstack(X_parts, format="csr", dtype=np.float32),
            np.concatenate(y_parts), np.concatenate(qid_parts), encoder)


def evaluate_stateaware_ranker(model, state_pack, encoder):
    rows = []
    n = state_pack["pack"]["n_ops"]
    for t in range(n - 1):
        candidates = np.arange(t+1, n, dtype=np.int64)
        Xq = stateaware_pair_matrix(state_pack, t, candidates, encoder)
        pred = model.predict(Xq)
        h1, h3, mrr = expected_tie_metrics(pred, target_position=0, tol=1e-8)
        rows.append({"position": t, "candidate_size": len(candidates),
                     "hits1": h1, "hits3": h3, "mrr": mrr})
    detail = pd.DataFrame(rows)
    return ({m: float(detail[m].mean()) for m in ["hits1", "hits3", "mrr"]}, detail)

# Keep results in one clearly named, separate directory (not E6).
OUT_STATE = Path("/content/battery_stateaware_xgb_lopo")
OUT_STATE.mkdir(parents=True, exist_ok=True)
PARTIAL = OUT_STATE / "stateaware_xgb_seed_partial.csv"
print("XGB params (frozen from E3):", XGB_PARAMS)
print("Expected fits:", len(chains) * len(SEEDS))

In [ ]:
# Full experiment: 10 held-out products x 3 seeds. Checkpoint after each fit.
# The in-fold X, y and qid are computed once per fold, and reused across seeds.
# To restart from scratch, delete the partial CSV explicitly before executing.
import gc
existing = pd.read_csv(PARTIAL) if PARTIAL.exists() else pd.DataFrame()
existing_keys = set(zip(existing.get("battery_id", []), existing.get("seed", [])))
completed = existing.to_dict("records") if len(existing) else []

for held_out in sorted(chains):
    todo = [seed for seed in SEEDS if (held_out, seed) not in existing_keys]
    if not todo:
        continue
    training = [b for b in sorted(chains) if b != held_out]
    vocab = build_fold_vocab(training)
    Xtr, ytr, qtr, enc = build_stateaware_training_data(training, vocab)
    test_pack = state_snapshots_for_battery(held_out, vocab)
    for seed in todo:
        start = time.time()
        model = build_xgb_model(seed)  # exact E3 parameterization
        model.fit(Xtr, ytr, qid=qtr, verbose=False)
        metrics, _ = evaluate_stateaware_ranker(model, test_pack, enc)
        row = {"battery_id": int(held_out), "battery": battery_names[held_out],
               "seed": int(seed), **metrics, "seconds": time.time() - start}
        completed.append(row)
        pd.DataFrame(completed).sort_values(["battery_id", "seed"]).to_csv(PARTIAL, index=False)
        print("Completed", row["battery"], "seed", seed,
              "Hits@1", round(metrics["hits1"], 5), "seconds", round(row["seconds"], 1))
        del model
    del Xtr, ytr, qtr, enc, test_pack
    gc.collect()

seed_results = pd.DataFrame(completed).sort_values(["battery_id", "seed"])
assert len(seed_results) == len(chains) * len(SEEDS)
assert not seed_results.duplicated(["battery_id", "seed"]).any()
seed_results.to_csv(OUT_STATE / "stateaware_xgb_seed_results.csv", index=False)
fold_mean = (seed_results.groupby(["battery_id", "battery"], as_index=False)
             .agg(hits1=("hits1", "mean"), hits3=("hits3", "mean"),
                  mrr=("mrr", "mean")))
fold_mean.to_csv(OUT_STATE / "stateaware_xgb_fold_mean.csv", index=False)
summary = fold_mean[["hits1", "hits3", "mrr"]].mean().to_frame("mean").T
summary.to_csv(OUT_STATE / "stateaware_xgb_aggregate.csv", index=False)
print("\nNew state-aware XGB product-macro means:")
display(summary)

## Descriptive comparison and the separately designated Hits@1 test

Use the **saved E6** `dynamic_real` product-level means from the pinned original repository, rather than re-training a slightly different DynamicReal and inadvertently mixing runs. These are *paired by held-out product*, not by random seed. The extra experiment's Hits@1 comparison is analyzed separately from the original four-comparison E6 Holm family.

This step runs only after all 30 fits; it does not tune the features or XGB parameters.

In [ ]:
import itertools
from io import StringIO
UPSTREAM_COMMIT = "21b931fc3d7e05ec0ae01ad8b032016fb7706969"

# Exact saved reference outputs from the E6 and E3 experiment-result ZIPs,
# originating from the pinned repository version. These are NOT newly trained results.
SAVED_REFERENCE_CSVS = {
    "results/extended/e6_state_aware/e6_fold_mean.csv": 'battery_id,battery,condition_summary,hits1,hits3,mrr,realization_sd_hits1\n1,Fiat 500e,dynamic_complete,0.1913660991570366,0.341516057832799,0.31576377719235826,0.0\n1,Fiat 500e,dynamic_noedge,0.16766624740749972,0.31798526713549474,0.29662952816027915,0.0\n1,Fiat 500e,dynamic_real,0.3684615147354206,0.5301728686269391,0.47931612546346064,0.0\n1,Fiat 500e,dynamic_shuffle_mean,0.21394425503438022,0.3723217735199566,0.34226782562536595,0.015443254759915663\n1,Fiat 500e,static_real,0.24457297630811206,0.42987163242998094,0.37800868808953353,0.0\n2,Chevrolet Spark,dynamic_complete,0.2549395358260041,0.4647900930782405,0.407036417339542,0.0\n2,Chevrolet Spark,dynamic_noedge,0.2064481529187928,0.39725292753311053,0.35646291186510887,0.0\n2,Chevrolet Spark,dynamic_real,0.4350671083375007,0.6445226822979313,0.5679026571507686,0.0\n2,Chevrolet Spark,dynamic_shuffle_mean,0.32516566663216684,0.5553705860575118,0.481192529926852,0.016485529767357218\n2,Chevrolet Spark,static_real,0.22929913205719707,0.41935115531996114,0.3786173428014774,0.0\n3,Hyundai IONIQ 5,dynamic_complete,0.15046410328442153,0.28031468521346375,0.2620557437947139,0.0\n3,Hyundai IONIQ 5,dynamic_noedge,0.11513531755137092,0.23575569358675527,0.22793140958277433,0.0\n3,Hyundai IONIQ 5,dynamic_real,0.3711227338977154,0.49115317578641027,0.46404047583867314,0.0\n3,Hyundai IONIQ 5,dynamic_shuffle_mean,0.3092456980569489,0.46287670269128434,0.42351442911809406,0.01081228171791683\n3,Hyundai IONIQ 5,static_real,0.20970953330057696,0.3537399988654613,0.32633503224042615,0.0\n4,BMW 530e,dynamic_complete,0.27619611133156957,0.4858424033506889,0.4244189761896191,0.0\n4,BMW 530e,dynamic_noedge,0.2589907641899167,0.4563778127396015,0.4038140362024234,0.0\n4,BMW 530e,dynamic_real,0.4008610577321848,0.6090322177825416,0.5316822036000647,0.0\n4,BMW 530e,dynamic_shuffle_mean,0.31561788441576305,0.5281204614567725,0.460939240395612,0.01942334481508904\n4,BMW 530e,static_real,0.33412235588201805,0.5570379805972215,0.4816561258367507,0.0\n5,Chevrolet Volt,dynamic_complete,0.25305435548868194,0.4377775242973711,0.3969526093880497,0.0\n5,Chevrolet Volt,dynamic_noedge,0.18604899989013712,0.3565071067654582,0.32871041041878274,0.0\n5,Chevrolet Volt,dynamic_real,0.4605136956777646,0.6455973119328923,0.5787715982993286,0.0\n5,Chevrolet Volt,dynamic_shuffle_mean,0.2931137832742303,0.4759143083233603,0.4322901828720449,0.01662023737136958\n5,Chevrolet Volt,static_real,0.22225812209323523,0.40857490175494,0.3623462874781927,0.0\n6,Volvo EX30,dynamic_complete,0.2695594806693917,0.4470382303679636,0.40052085966821754,0.0\n6,Volvo EX30,dynamic_noedge,0.2660946738712516,0.45622052955026277,0.4022828781831403,0.0\n6,Volvo EX30,dynamic_real,0.36973017857818485,0.5517831283271473,0.4938757082248835,0.0\n6,Volvo EX30,dynamic_shuffle_mean,0.3051255953307445,0.48380535742080494,0.43219384935124494,0.022993419097663278\n6,Volvo EX30,static_real,0.3440952579432642,0.5226296891737081,0.47154441226708155,0.0\n7,BMW 330e,dynamic_complete,0.3086566802750906,0.5120811519363827,0.4525573291927493,0.0\n7,BMW 330e,dynamic_noedge,0.2864805230989334,0.48358686844209925,0.43172726655093935,0.0\n7,BMW 330e,dynamic_real,0.388604787723198,0.5844041067593376,0.518676768134725,0.0\n7,BMW 330e,dynamic_shuffle_mean,0.3340040551224654,0.531841225196456,0.4729653449912725,0.013332170779418326\n7,BMW 330e,static_real,0.35003641165482186,0.552245987101218,0.4907716974465007,0.0\n8,MG ZS,dynamic_complete,0.14788819719371615,0.29275686865507694,0.27118924371001435,0.0\n8,MG ZS,dynamic_noedge,0.12997242310983276,0.2740830865228384,0.2529125652669573,0.0\n8,MG ZS,dynamic_real,0.35818341002521703,0.5230834727984962,0.4711861291816195,0.0\n8,MG ZS,dynamic_shuffle_mean,0.17169882046156482,0.3191153601436544,0.2951884791312241,0.0232258195738712\n8,MG ZS,static_real,0.20498481390073497,0.37224929389384614,0.33254304903963355,0.0\n9,Mitsubishi Outlander,dynamic_complete,0.2907498966534204,0.5163874965980679,0.44451485958484643,0.0\n9,Mitsubishi Outlander,dynamic_noedge,0.26665551432769896,0.5008051780322583,0.42877698611719306,0.0\n9,Mitsubishi Outlander,dynamic_real,0.36721054396894814,0.599189629464842,0.5144977217213876,0.0\n9,Mitsubishi Outlander,dynamic_shuffle_mean,0.3024492602381819,0.508745840326891,0.4498614226605362,0.011605847810527738\n9,Mitsubishi Outlander,static_real,0.3310562846943881,0.5690737032022992,0.4904282808584801,0.0\n10,BMW i3,dynamic_complete,0.19048772791384336,0.3617125768978999,0.3296451387330712,0.0\n10,BMW i3,dynamic_noedge,0.1629464998899922,0.3182968640110619,0.30053494134216224,0.0\n10,BMW i3,dynamic_real,0.3831840011088925,0.5497227305322886,0.5018918492969214,0.0\n10,BMW i3,dynamic_shuffle_mean,0.3751847393887006,0.5617945282436212,0.5025006800376789,0.014391954990366157\n10,BMW i3,static_real,0.3208962565537061,0.47964613400220363,0.4427088416084403,0.0\n',
    "results/extended/e3_strong_baselines/e3_fold_all_methods.csv": 'battery_id,battery,method,hits1,hits3,mrr,random_hits1,hits1_skill_score\n1,Fiat 500e,XGBRanker,0.17517652668759656,0.3252464455432601,0.29788420851339226,0.017387186537289944,0.16058139888717696\n2,Chevrolet Spark,XGBRanker,0.23303307830611333,0.4167116291774409,0.37968047061689864,0.034120961085443983,0.20593895219447408\n3,Hyundai IONIQ 5,XGBRanker,0.12374565264332267,0.25847069425696856,0.24583961401473423,0.016928362206878683,0.10865666990071657\n4,BMW 530e,XGBRanker,0.2828735869580898,0.4904138050558486,0.429493648105683,0.035902183457577015,0.25616840870590785\n5,Chevrolet Volt,XGBRanker,0.20554364513779436,0.37469653112958134,0.34618558606805977,0.03429043661034367,0.1773340712567339\n6,Volvo EX30,XGBRanker,0.2823116050881828,0.4822258205555538,0.4192757540791647,0.02822239170647377,0.2614684792211853\n7,BMW 330e,XGBRanker,0.2928907795091898,0.4808090906643215,0.43518870386710556,0.041912241063584425,0.26195777589750185\n8,MG ZS,XGBRanker,0.1277595963564069,0.2727029707100346,0.25208789419861405,0.0194761593530702,0.11043427249243984\n9,Mitsubishi Outlander,XGBRanker,0.23010578759940084,0.446534867793903,0.381927306210895,0.03178003895018658,0.2048354264811637\n10,BMW i3,XGBRanker,0.17325934595865222,0.3317935417519257,0.311223702687956,0.033300083149016065,0.14478046430949554\n1,Fiat 500e,ClassTransitionPrior,0.052693714721298454,0.11775137589295064,0.12503924080529702,0.017387186537289944,0.035931271911251526\n1,Fiat 500e,GraphDistance,0.07258683692704314,0.1936963396581348,0.18419437498950084,0.017387186537289944,0.05617639993440612\n1,Fiat 500e,SharedComponent,0.19921219724007966,0.3717007628432336,0.3268982020748325,0.017387186537289944,0.18504237702950527\n1,Fiat 500e,UniformRandom,0.017387186537289944,0.04547706763326021,0.05872830570763799,0.017387186537289944,0.0\n2,Chevrolet Spark,ClassTransitionPrior,0.15387397227574984,0.31291932201687034,0.28692898716186455,0.034120961085443983,0.12398344550978446\n2,Chevrolet Spark,GraphDistance,0.0965155710117637,0.24164252141852458,0.22859649096935092,0.034120961085443983,0.06459878246911546\n2,Chevrolet Spark,SharedComponent,0.24136203330416345,0.4516310100921312,0.3920695147746284,0.034120961085443983,0.21456213860031026\n2,Chevrolet Spark,UniformRandom,0.034120961085443983,0.08739282337609243,0.10212111643111046,0.034120961085443983,0.0\n3,Hyundai IONIQ 5,ClassTransitionPrior,0.07131469457889504,0.15239399738089926,0.15143967293928878,0.016928362206878683,0.05532285774626475\n3,Hyundai IONIQ 5,GraphDistance,0.09996046540963026,0.27792463808905843,0.23673796336871902,0.016928362206878683,0.08446190492195335\n3,Hyundai IONIQ 5,SharedComponent,0.24376195957803728,0.38548030261765504,0.35455551955180936,0.016928362206878683,0.23073964159964272\n3,Hyundai IONIQ 5,UniformRandom,0.016928362206878683,0.04430840268281223,0.057435295430656015,0.016928362206878683,0.0\n4,BMW 530e,ClassTransitionPrior,0.15306641606811633,0.2880209042553044,0.26676764320242685,0.035902183457577015,0.12152732907406576\n4,BMW 530e,GraphDistance,0.08470866244033767,0.21590942681145886,0.20807496818004712,0.035902183457577015,0.050623990787363266\n4,BMW 530e,SharedComponent,0.24377534782132793,0.4542559797697162,0.39180528817099686,0.035902183457577015,0.21561418436694896\n4,BMW 530e,UniformRandom,0.035902183457577015,0.09178298349375014,0.10640231518672419,0.035902183457577015,0.0\n5,Chevrolet Volt,ClassTransitionPrior,0.09492653863330777,0.1944181701167908,0.21669332654591542,0.03429043661034367,0.06278917007938742\n5,Chevrolet Volt,GraphDistance,0.11291008572789929,0.2965615824849027,0.2595103981347982,0.03429043661034367,0.0814112773633507\n5,Chevrolet Volt,SharedComponent,0.2925009762264311,0.5552137720527874,0.45125495560904794,0.03429043661034367,0.2673790851876461\n5,Chevrolet Volt,UniformRandom,0.03429043661034367,0.08781106886717559,0.10253075592114458,0.03429043661034367,0.0\n6,Volvo EX30,ClassTransitionPrior,0.12624667134467765,0.2770733473673663,0.2424199248832496,0.02822239170647377,0.10087110343110064\n6,Volvo EX30,GraphDistance,0.0622795157602506,0.16193986440334554,0.1662285201724685,0.02822239170647377,0.035046211975991365\n6,Volvo EX30,SharedComponent,0.21273669012541138,0.37807726971071226,0.34071577770480294,0.02822239170647377,0.18987296768748443\n6,Volvo EX30,UniformRandom,0.02822239170647377,0.07276241321465941,0.08753805509809946,0.02822239170647377,0.0\n7,BMW 330e,ClassTransitionPrior,0.18295858484164146,0.3679694386186085,0.3315743842273448,0.041912241063584425,0.14721651796765906\n7,BMW 330e,GraphDistance,0.07923903342103578,0.19925556180156886,0.19685685538341235,0.041912241063584425,0.03895967985113207\n7,BMW 330e,SharedComponent,0.24363652044275544,0.4386018690205741,0.3879776967710354,0.041912241063584425,0.2105488536907177\n7,BMW 330e,UniformRandom,0.041912241063584425,0.10650595395998406,0.12047853400200627,0.041912241063584425,0.0\n8,MG ZS,ClassTransitionPrior,0.07387794227230506,0.1724493212103912,0.1696243258000834,0.0194761593530702,0.055482366327107006\n8,MG ZS,GraphDistance,0.05027461807465722,0.13706238633406337,0.1412977801764832,0.0194761593530702,0.03141020895653778\n8,MG ZS,SharedComponent,0.18639008656104056,0.3588644492855682,0.31241703221638034,0.0194761593530702,0.17022934097945433\n8,MG ZS,UniformRandom,0.0194761593530702,0.05078321812037268,0.06452946889165809,0.0194761593530702,0.0\n9,Mitsubishi Outlander,ClassTransitionPrior,0.13460740173341043,0.2768991282771544,0.24973383995779777,0.03178003895018658,0.10620248179115317\n9,Mitsubishi Outlander,GraphDistance,0.06407664439637763,0.17849366945286912,0.17762730801130935,0.03178003895018658,0.03335668210266266\n9,Mitsubishi Outlander,SharedComponent,0.23386954520545317,0.38842182242954637,0.35049810117629854,0.03178003895018658,0.20872272250630597\n9,Mitsubishi Outlander,UniformRandom,0.03178003895018658,0.08160385311429598,0.09641134591861046,0.03178003895018658,0.0\n10,BMW i3,ClassTransitionPrior,0.10061108433941368,0.20335941580893876,0.2024044761774987,0.033300083149016065,0.06962967516296328\n10,BMW i3,GraphDistance,0.0703407400289034,0.1848594293890358,0.19318623208311544,0.033300083149016065,0.03831660294390729\n10,BMW i3,SharedComponent,0.3076146298796843,0.46644854080184367,0.4278704626381562,0.033300083149016065,0.28376390847766425\n10,BMW i3,UniformRandom,0.033300083149016065,0.08536536572611797,0.10012994933104356,0.033300083149016065,0.0\n',
}

def read_pinned_csv(repo_relpath):
    assert repo_relpath in SAVED_REFERENCE_CSVS, repo_relpath
    return pd.read_csv(StringIO(SAVED_REFERENCE_CSVS[repo_relpath]))

canonical_dynamic = read_pinned_csv("results/extended/e6_state_aware/e6_fold_mean.csv")
canonical_dynamic = canonical_dynamic[
    canonical_dynamic["condition_summary"] == "dynamic_real"
][["battery_id", "battery", "hits1", "hits3", "mrr"]]
assert len(canonical_dynamic) == 10
assert np.isclose(canonical_dynamic["hits1"].mean(), 0.390294, atol=1e-5)

paired = canonical_dynamic.merge(fold_mean, on="battery_id", how="inner",
                                 suffixes=("_dynamic_real", "_stateaware_xgb"),
                                 validate="one_to_one")
assert len(paired) == 10
assert (paired["battery_dynamic_real"] == paired["battery_stateaware_xgb"]).all()
for metric in ["hits1", "hits3", "mrr"]:
    paired[f"difference_{metric}"] = (paired[f"{metric}_dynamic_real"] -
                                      paired[f"{metric}_stateaware_xgb"])


def exact_signflip_two_sided(diffs):
    diffs = np.asarray(diffs, dtype=float)
    observed = abs(diffs.mean())
    # All 2^10 signs on 10 independent held-out product differences.
    signs = np.asarray(list(itertools.product([-1., 1.], repeat=len(diffs))))
    null_means = np.abs((signs * diffs[None, :]).mean(axis=1))
    return float(np.mean(null_means >= observed - 1e-12))


def descriptive_product_bootstrap(diffs, seed=20261011, replicates=20000):
    arr = np.asarray(diffs, dtype=float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(arr), size=(replicates, len(arr)))
    return np.percentile(arr[idx].mean(axis=1), [2.5, 97.5])

primary = paired["difference_hits1"].to_numpy(float)
ci = descriptive_product_bootstrap(primary)
contrast = pd.DataFrame([{
    "contrast": "DynamicReal - StateAwareXGBRanker",
    "mean_difference": primary.mean(), "descriptive_ci_low": ci[0],
    "descriptive_ci_high": ci[1],
    "positive_products": int((primary > 0).sum()),
    "negative_products": int((primary < 0).sum()),
    "zero_products": int((primary == 0).sum()),
    "p_exact_two_sided": exact_signflip_two_sided(primary),
    "multiplicity_family": "separate additional prespecified Hits@1 contrast",
}])
paired.to_csv(OUT_STATE / "stateaware_xgb_vs_dynamicreal_products.csv", index=False)
contrast.to_csv(OUT_STATE / "stateaware_xgb_vs_dynamicreal_contrast.csv", index=False)
print("Product differences (DynamicReal minus new baseline):")
display(paired[["battery_dynamic_real", "difference_hits1", "difference_hits3", "difference_mrr"]])
print("Designated Hits@1 contrast:")
display(contrast)

static_xgb = read_pinned_csv("results/extended/e3_strong_baselines/e3_fold_all_methods.csv")
static_xgb = static_xgb[static_xgb["method"] == "XGBRanker"]
assert len(static_xgb) == 10
static_baseline_mean = static_xgb[["hits1", "hits3", "mrr"]].mean()
print("Static XGBRanker means (existing E3):", static_baseline_mean.to_dict())
print("State-aware XGBRanker means:", fold_mean[["hits1", "hits3", "mrr"]].mean().to_dict())

metadata = {
    "base_code_commit": UPSTREAM_COMMIT,
    "primary_contrast": "DynamicReal - StateAwareXGBRanker Hits@1",
    "statistical_unit": "held-out battery product (10)",
    "train_seeds": SEEDS,
    "xgb_parameters": XGB_PARAMS,
    "node_state_features": STATE_NODE_FEATURES,
    "global_features": GLOBAL_STATE_FEATURES,
    "ranking_task": "reference next-operation ranking",
    "state_timing": "post-current operation; executed prefix only",
    "note": "Additional experiment, kept separate from the canonical E6 Holm family",
}
(OUT_STATE / "stateaware_xgb_metadata.json").write_text(
    json.dumps(metadata, indent=2, default=str), encoding="utf-8")
print("Saved outputs to", OUT_STATE)

In [ ]:
# Package complete outputs for sharing and independent inspection.
from zipfile import ZipFile, ZIP_DEFLATED
archive_path = Path("/content/stateaware_xgb_results.zip")
with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
    for file in sorted(OUT_STATE.glob("*")):
        if file.is_file():
            archive.write(file, arcname=file.name)
print("Share this archive for the next manuscript step:", archive_path)

### Interpretation rules to preserve after the run

- The baseline's comparison with DynamicReal does **not** establish that physical topology is irrelevant if the difference is small. The state-aware tabular features themselves use information derived from the physical fixation graph.
- Do not use held-out results to change the feature recipe, hyperparameters, or primary contrast. If a code defect is found, document and rerun every affected fold/seed consistently.
- The existing DynamicReal–DynamicShuffled inference is a separate, already completed, mechanism comparison and should not be invalidated or overstated based on this baseline.
- No statistics in this draft notebook are experimental findings until the external data are supplied and the notebook is executed.